# Chapter 31
## ING Rhythms

In [ ]:
import brian2 as b2
import matplotlib.pyplot as plt
import numpy as np


def validate_probability(name, value):
    value = float(value)
    if not 0.0 <= value <= 1.0:
        raise ValueError(f"{name} must be in [0, 1], got {value}")
    return value


def tau_peak(tau_d_ms, tau_r_ms, tau_dq_ms, dt_ms=0.01):
    s = 0.0
    t = 0.0
    ds = np.exp(-t / tau_dq_ms) * (1.0 - s) / tau_r_ms - s / tau_d_ms
    while ds > 0.0:
        t_old, ds_old = t, ds
        s_mid = s + 0.5 * dt_ms * ds
        ds_mid = (
            np.exp(-(t + 0.5 * dt_ms) / tau_dq_ms)
            * (1.0 - s_mid) / tau_r_ms
            - s_mid / tau_d_ms
        )
        s += dt_ms * ds_mid
        t += dt_ms
        ds = np.exp(-t / tau_dq_ms) * (1.0 - s) / tau_r_ms - s / tau_d_ms
    return (t_old * (-ds) + t * ds_old) / (ds_old - ds)


def solve_tau_dq(tau_d_ms, tau_r_ms, tau_peak_ms):
    left = 1.0
    while tau_peak(tau_d_ms, tau_r_ms, left) > tau_peak_ms:
        left *= 0.5
    right = tau_r_ms
    while tau_peak(tau_d_ms, tau_r_ms, right) < tau_peak_ms:
        right *= 2.0
    while right - left > 1e-12:
        middle = 0.5 * (left + right)
        if tau_peak(tau_d_ms, tau_r_ms, middle) <= tau_peak_ms:
            left = middle
        else:
            right = middle
    return 0.5 * (left + right)


In [ ]:
WB_EQS = """
dv/dt = (g_l*(E_l-v) + g_k*n**4*(E_k-v)
         + g_na*m_inf**3*h*(E_na-v) + i_ext + I_chem + I_gap)/C : volt
dh/dt = (h_inf-h)/tau_h : 1
dn/dt = (n_inf-n)/tau_n : 1
dq/dt = 0.5*(1+tanh(v/(10*mV)))*(1-q)/(0.1*ms) - q/tau_dq : 1
ds/dt = q*(1-s)/tau_r - s/tau_d : 1
m_inf = alpha_m/(alpha_m+beta_m) : 1
h_inf = alpha_h/(alpha_h+beta_h) : 1
n_inf = alpha_n/(alpha_n+beta_n) : 1
alpha_m = 0.1/mV*(v+35*mV)/(1-exp(-(v+35*mV)/(10*mV)))/ms : Hz
beta_m = 4*exp(-(v+60*mV)/(18*mV))/ms : Hz
alpha_h = 0.07*exp(-(v+58*mV)/(20*mV))/ms : Hz
beta_h = 1/(exp(-(v+28*mV)/(10*mV))+1)/ms : Hz
alpha_n = -0.01/mV*(v+34*mV)/(exp(-(v+34*mV)/(10*mV))-1)/ms : Hz
beta_n = 0.125*exp(-(v+44*mV)/(80*mV))/ms : Hz
tau_h = 1/(5*(alpha_h+beta_h)) : second
tau_n = 1/(5*(alpha_n+beta_n)) : second
i_ext : amp
I_chem : amp
I_gap : amp
tau_dq : second
tau_r : second
tau_d : second
C : farad (constant)
g_l : siemens (constant)
g_k : siemens (constant)
g_na : siemens (constant)
E_l : volt (constant)
E_k : volt (constant)
E_na : volt (constant)
"""


def initial_wb_state(count, mode, rng):
    if mode == "uniform_random":
        return {
            "v": rng.uniform(-100.0, 50.0, size=count) * b2.mV,
            "h": rng.uniform(0.0, 1.0, size=count),
            "n": rng.uniform(0.0, 1.0, size=count),
            "q": np.zeros(count),
            "s": np.zeros(count),
        }
    if mode == "fixed":
        return {
            "v": np.full(count, -75.0) * b2.mV,
            "h": np.full(count, 0.1),
            "n": np.full(count, 0.1),
            "q": np.zeros(count),
            "s": np.zeros(count),
        }
    raise ValueError(f"unknown WB initial-state mode: {mode}")


## 1_CELL_ING

In [ ]:
def simulate_single_cell_ing(
    i_ext=1.5 * b2.uA,
    g_ii=0.5 * b2.msiemens,
    tau_d=9 * b2.ms,
    duration=200 * b2.ms,
    dt=0.01 * b2.ms,
    record=True,
):
    b2.start_scope()
    b2.defaultclock.dt = dt

    cells = b2.NeuronGroup(
        1, WB_EQS, method="rk2", threshold="v > -20*mV", refractory="v > -20*mV"
    )
    cells.C = 1.0 * b2.ufarad
    cells.g_l = 0.1 * b2.msiemens
    cells.g_k = 9.0 * b2.msiemens
    cells.g_na = 35.0 * b2.msiemens
    cells.E_l = -65.0 * b2.mV
    cells.E_k = -90.0 * b2.mV
    cells.E_na = 55.0 * b2.mV
    cells.i_ext = i_ext
    cells.tau_dq = solve_tau_dq(float(tau_d / b2.ms), 0.5, 0.5) * b2.ms
    cells.tau_r = 0.5 * b2.ms
    cells.tau_d = tau_d
    cells.v = -75.0 * b2.mV
    cells.h = 0.1
    cells.n = 0.1
    cells.q = 0.0
    cells.s = 0.0

    chem = b2.Synapses(
        cells,
        cells,
        model="g : siemens\nI_chem_post = g*s_pre*(-75*mV-v_post) : amp (summed)",
    )
    chem.connect(i=[0], j=[0])
    chem.g = g_ii

    state_monitor = b2.StateMonitor(cells, "v", record=record)
    spike_monitor = b2.SpikeMonitor(cells)
    b2.run(duration)

    spike_ms = np.asarray(spike_monitor.t / b2.ms)
    if len(spike_ms) < 2:
        raise RuntimeError("fewer than two spikes")
    period_ms = float(spike_ms[-1] - spike_ms[-2])
    if record:
        t_ms = np.asarray(state_monitor.t / b2.ms)
        v_mv = np.asarray(state_monitor.v[0] / b2.mV)
    else:
        t_ms = np.array([])
        v_mv = np.array([])
    return {
        "t_ms": t_ms,
        "v_mv": v_mv,
        "spike_ms": spike_ms,
        "period_ms": period_ms,
        "frequency_hz": 1000.0 / period_ms,
    }


if __name__ == "__main__":
    one_cell = simulate_single_cell_ing()
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.plot(one_cell["t_ms"], one_cell["v_mv"], color="k", lw=2)
    ax.set(xlabel="time [ms]", ylabel="v [mV]", ylim=(-100, 50))
    fig.tight_layout()


In [ ]:
def compute_ing_condition_numbers(duration=200*b2.ms):
    base = simulate_single_cell_ing(duration=duration, record=False)
    reduced_i = simulate_single_cell_ing(i_ext=1.5*0.99*b2.uA, duration=duration, record=False)
    raised_g = simulate_single_cell_ing(g_ii=0.5*1.01*b2.msiemens, duration=duration, record=False)
    raised_tau = simulate_single_cell_ing(tau_d=9*1.01*b2.ms, duration=duration, record=False)
    period = base["period_ms"]
    return {
        "base_period_ms": period,
        "pct_i_ext": 100.0*(period-reduced_i["period_ms"])/period,
        "pct_g_ii": 100.0*(period-raised_g["period_ms"])/period,
        "pct_tau_d": 100.0*(period-raised_tau["period_ms"])/period,
    }


if __name__ == "__main__":
    condition_numbers = compute_ing_condition_numbers()
    print("base_period", condition_numbers["base_period_ms"])
    print("pct_i_ext", condition_numbers["pct_i_ext"])
    print("pct_g_ii", condition_numbers["pct_g_ii"])
    print("pct_tau_d", condition_numbers["pct_tau_d"])
